In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from mlp_model.mlp import MLP
from mlp_model.layers.hidden import Hidden
from mlp_model.layers.output_layer import OutputLayer
from mlp_model.optimizers.sgd import SGDOptimizer
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.losses.bce_loss import BCELoss
from mlp_model.activations.relu import ReLU
from mlp_model.activations.tanh import Tanh
from mlp_model.activations.softmax import Softmax
from mlp_model.activations.sigmoid import Sigmoid
from mlp_model.initialize_weights.xavier import XavierInitializer
from trainer.trainer import MLPTrainer

import visualisation.classification_visualise as cl_vs

In [ ]:
data_kind = "simple" # simple, three_gauss
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/classification/train" / f"data.{data_kind}.train.{data_count}.csv"
test_filepath = Path(project_root) / "data/classification/test" / f"data.{data_kind}.test.{data_count}.csv"

In [ ]:
df = pd.read_csv(train_filepath)
df.head(5)

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
X = df.drop(["cls"], axis=1).to_numpy(dtype=float)
y = (df["cls"] - 1).to_numpy().reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, 
    stratify=y.reshape(-1), random_state=42)

In [ ]:
plt.figure(figsize=(8, 6))

for cls in np.unique(y):
    mask = y_train.reshape(-1) == cls

    plt.scatter(
        X_train[mask, 0],
        X_train[mask, 1],
        label=f"Class {cls}",
        alpha=0.7,
        s=20
    )

In [ ]:
df_test = pd.read_csv(test_filepath)

X_test = df.drop(["cls"], axis=1).to_numpy(dtype=float)
y_test = (df["cls"] - 1).to_numpy().reshape(-1, 1)

In [ ]:
def create_mlp_model(input_size, classes_count, 
                     hidden_sizes=(16,), activation_hidden=Sigmoid(), random_state=42):
    layers = []
    curr_size = input_size

    for size in hidden_sizes:
        layers.append(Hidden(curr_size, size, activation_hidden))
        curr_size = size

    if classes_count == 2:
        layers.append(OutputLayer(curr_size, 1, Sigmoid()))
    else:
        layers.append(OutputLayer(curr_size, classes_count, Softmax()))

    model = MLP(layers)
    model.initialize_weights(XavierInitializer(random_state))

    return model

In [ ]:
def run_training(activation_hidden, loss_cls, hidden_sizes=(16,), classes_count=2, 
                 seeds=(42, 43, 44, 45, 46), epochs=100, batch_size=32, learning_rate=0.01):
    runs = []

    for seed in seeds:
        np.random.seed(seed)

        model = create_mlp_model(X_train.shape[1], classes_count, hidden_sizes, activation_hidden)
        optimizer = SGDOptimizer(model, learning_rate)

        trainer = MLPTrainer(model, optimizer, loss_cls, epochs, batch_size, 42)

        history = trainer.fit(X_train, y_train, X_val, y_val)

        result = {
            "seed": seed,
            "loss": history["val_loss"][-1],
            "metrics": history["val_metrics"][-1],
            "history": history,
            "trainer": trainer
        }

        runs.append(result)

    return runs

In [ ]:
def summarize_results(trainings):
    summary = []

    for name, runs in trainings.items():
        row = {"config": name}

        losses = [run["loss"] for run in runs]

        row["loss_mean"] = np.mean(losses)
        row["loss_std"] = np.std(losses)

        metrics_names = runs[0]["metrics"].keys()

        for metric in metrics_names:
            values = [run["metrics"][metric] for run in runs]

            row[f"{metric}_mean"] = np.mean(values)
            row[f"{metric}_std"] = np.std(values)

        summary.append(row)

    return pd.DataFrame(summary)

In [ ]:
def evaluate_runs(runs, X_test, y_test):
    results = []

    for run in runs:
        trainer = run["trainer"]

        test_loss, test_metrics = trainer.evaluate(
            X_test, y_test
        )

        results.append({"seed": run["seed"], "test_loss": test_loss, **test_metrics})

    return pd.DataFrame(results)

## Rodzaje funkcji aktywacji w warstwie ukrytej

In [ ]:
activations = {
    "Sigmoid": Sigmoid(),
    "ReLU": ReLU(),
    "Tanh": Tanh()
}

trainings = {}

for name, activation in activations.items():
    trainings[name] = run_training(hidden_sizes=(16,), activation_hidden=activation, 
                                   epochs=50, loss_cls=BCELoss())

    cl_vs.plot_decision_boundary()

display(summarize_results(trainings))

In [ ]:
test_results = evaluate_runs(trainings["Tanh"], X_test, y_test)
print("Tanh test results: ")
display(test_results)
print("Tanh average test results: ")
display(test_results[["test_loss", "accuracy", "precision", "f1_score"]
].agg(["mean", "std"]))